# New analysis 01 — Steam Games Dataset: loading, label audit, pre-release features

**Why a new dataset:** see `../../DATASET_SWITCH_PROPOSAL.md`. The VGChartz data had almost no
information that exists before launch, so the honest pre-release model in
`old_analysis/week3/05` could only lean on past franchise/publisher sales.

**Dataset:** Kaggle [`fronkongames/steam-games-dataset`](https://www.kaggle.com/datasets/fronkongames/steam-games-dataset)
— Steam store pages plus SteamSpy owner estimates.

**This notebook fits no models.** It:
1. loads the data and documents two defects in the files,
2. audits the target (`estimated_owners`) — when it is valid and when it is missing,
3. classifies every column as *pre-release*, *post-release* (leakage), or *uncertain*,
4. builds a pre-release feature table (including developer/publisher history on an expanding window),
5. checks that the features carry signal, and saves the table for modeling.

The leakage discipline from `old_analysis/week3` carries over unchanged.

In [1]:
import json
import os
import re

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import kagglehub

OUT_DIR = os.path.join("..", "outputs")
DATA_DIR = os.path.join("..", "data", "processed")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# Games released after this date have not had ~12 months on sale before the owner
# estimates stopped updating (see Section 3). Excluded from the modeling set.
RELEASE_CUTOFF = pd.Timestamp("2024-06-30")
HIT_OWNERS = 100_000  # "hit" = SteamSpy lower bound >= 100k owners

C:\Users\Nobutaka\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load

Downloaded with `kagglehub` (cached after the first run, ~300 MB compressed). The dataset ships
as `games.csv` and `games.json`. **This notebook uses the JSON**, because the CSV has two defects:

- its header merges two columns into one name (`"DiscountDLC count"`), so a naive `read_csv`
  shifts every value one column to the left (e.g. `AppID` holds the game name), and
- it has fewer rows than the JSON and drops fields the JSON keeps (`short_description`,
  `packages`, tag vote counts).

In [2]:
path = kagglehub.dataset_download("fronkongames/steam-games-dataset")
print("Path to dataset files:", path)

with open(os.path.join(path, "games.json"), encoding="utf-8") as f:
    raw = json.load(f)
print(f"Apps in games.json: {len(raw):,}")

csv_rows = sum(1 for _ in open(os.path.join(path, "games.csv"), encoding="utf-8")) - 1
csv_header = pd.read_csv(os.path.join(path, "games.csv"), nrows=0).columns.tolist()
print(f"Lines in games.csv: {csv_rows:,} (approx; descriptions contain newlines)")
print("Merged CSV header present:", "DiscountDLC count" in csv_header)

Path to dataset files: C:\Users\Nobutaka\.cache\kagglehub\datasets\fronkongames\steam-games-dataset\versions\49


Apps in games.json: 141,900


Lines in games.csv: 125,855 (approx; descriptions contain newlines)
Merged CSV header present: True


Flatten to one row per app. List-valued fields are kept as lists for now; counts are derived
below. `movies` is extracted to check coverage, even though it turns out to be empty.

In [3]:
def first_package_text(v):
    for p in v.get("packages") or []:
        for s in p.get("subs") or []:
            if s.get("text"):
                return s["text"]
    return None


records = []
for appid, v in raw.items():
    records.append({
        "appid": int(appid),
        "name": v.get("name"),
        "release_date": v.get("release_date"),
        "estimated_owners": v.get("estimated_owners"),
        "price_current": v.get("price"),
        "package_text": first_package_text(v),
        "required_age": v.get("required_age"),
        "dlc_count": v.get("dlc_count"),
        "short_description": v.get("short_description") or "",
        "detailed_description_len": len(v.get("detailed_description") or ""),
        "website": v.get("website") or "",
        "support_email": v.get("support_email") or "",
        "windows": v.get("windows"), "mac": v.get("mac"), "linux": v.get("linux"),
        "metacritic_score": v.get("metacritic_score"),
        "achievements": v.get("achievements"),
        "recommendations": v.get("recommendations"),
        "supported_languages": v.get("supported_languages") or [],
        "full_audio_languages": v.get("full_audio_languages") or [],
        "developers": v.get("developers") or [],
        "publishers": v.get("publishers") or [],
        "categories": v.get("categories") or [],
        "genres": v.get("genres") or [],
        "n_screenshots": len(v.get("screenshots") or []),
        "n_movies": len(v.get("movies") or []),
        "user_score": v.get("user_score"),
        "positive": v.get("positive"), "negative": v.get("negative"),
        "peak_ccu": v.get("peak_ccu"),
        "average_playtime_forever": v.get("average_playtime_forever"),
        "discount": v.get("discount"),
        "tags": v.get("tags") or {},
    })
del raw

df = pd.DataFrame(records)
df["release_dt"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")
print(df.shape)
print("Unparseable release dates:", df["release_dt"].isna().sum())
print(f"Apps with any trailer in `movies`: {(df['n_movies'] > 0).sum()}  -> trailer features unavailable")

(141900, 34)
Unparseable release dates: 0
Apps with any trailer in `movies`: 0  -> trailer features unavailable


## 2. The target: SteamSpy `estimated_owners`

Owners are reported as ranges. Parse the lower/upper bounds.

In [4]:
bounds = df["estimated_owners"].str.extract(r"(\d+)\s*-\s*(\d+)").astype(float)
df["owners_low"], df["owners_high"] = bounds[0], bounds[1]

print(df["estimated_owners"].value_counts().reindex(
    df.drop_duplicates("estimated_owners").sort_values("owners_low")["estimated_owners"]).to_string())

estimated_owners
0 - 0                    38990
0 - 20000                77328
20000 - 50000            11400
50000 - 100000            5359
100000 - 200000           3457
200000 - 500000           2857
500000 - 1000000          1155
1000000 - 2000000          729
2000000 - 5000000          405
5000000 - 10000000         125
10000000 - 20000000         51
20000000 - 50000000         31
50000000 - 100000000         9
100000000 - 200000000        4


## 3. Label audit — `"0 - 0"` means *no data*, not *no sales*

About one app in five is `"0 - 0"`. If that meant "zero owners" it would be a valid (bottom)
class; if it means "SteamSpy has no record", it is a missing label. Evidence:

In [5]:
zero = df["estimated_owners"] == "0 - 0"
reviews = df["positive"] + df["negative"]

evidence = pd.DataFrame({
    "0 - 0": [
        (reviews[zero] > 0).mean(), (df.loc[zero, "peak_ccu"] > 0).mean(),
        (df.loc[zero, "average_playtime_forever"] > 0).mean(), (df.loc[zero, "tags"].str.len() > 0).mean(),
    ],
    "0 - 20000": [
        (reviews[df["estimated_owners"] == "0 - 20000"] > 0).mean(),
        (df.loc[df["estimated_owners"] == "0 - 20000", "peak_ccu"] > 0).mean(),
        (df.loc[df["estimated_owners"] == "0 - 20000", "average_playtime_forever"] > 0).mean(),
        (df.loc[df["estimated_owners"] == "0 - 20000", "tags"].str.len() > 0).mean(),
    ],
}, index=["has any review", "peak CCU > 0", "playtime > 0", "has tags"]).round(3)
evidence

,0 - 0,0 - 20000
has any review,0.0,0.743
peak CCU > 0,0.0,0.101
playtime > 0,0.0,0.126
has tags,0.0,0.748


Every SteamSpy-sourced field is empty for `"0 - 0"` apps, while the smallest real class
(`0 - 20000`) mostly has reviews, tags and playtime. `"0 - 0"` is a **missing label** — the app was
never picked up by SteamSpy. Now look at *when* it happens:

In [6]:
by_month = (
    df[df["release_dt"] >= "2019-01-01"]
    .groupby(df["release_dt"].dt.to_period("Q"))
    .agg(apps=("appid", "size"), share_no_data=("estimated_owners", lambda s: (s == "0 - 0").mean()))
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(by_month.index.astype(str), by_month["share_no_data"], marker="o")
ax.axvline(str(RELEASE_CUTOFF.to_period("Q")), color="grey", ls="--", label=f"release cutoff {RELEASE_CUTOFF.date()}")
ax.set_ylabel('share of apps with "0 - 0" owners')
ax.set_title("Missing owner labels by release quarter")
ax.tick_params(axis="x", rotation=60)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "01_missing_labels_by_quarter.png"), dpi=150)
print("Saved outputs/01_missing_labels_by_quarter.png")
by_month.round(3).tail(12)

Saved outputs/01_missing_labels_by_quarter.png


,apps,share_no_data
release_dt,,
2024Q1,4356,0.212
2024Q2,4767,0.235
2024Q3,5109,0.237
2024Q4,5810,0.212
2025Q1,5397,0.246
2025Q2,6132,0.320
2025Q3,6563,0.498
2025Q4,6884,0.543
2026Q1,5637,0.894


Two findings that shape the whole project:

1. **Labels stop updating around mid-2025.** The no-data share jumps from ~25% to ~50% in
   mid-2025 and to ~90% in 2026. Games released after that were never measured.
2. **Censoring.** Owners accumulate over time, so a game released a month before the snapshot
   looks like a flop even if it will sell well. The model must only be trained and tested on
   games that had a comparable time on sale.

**Decision:** keep only games released on or before `RELEASE_CUTOFF` (2024-06-30, roughly 12
months before labels stop updating), and drop `"0 - 0"` rows as unlabeled.

*Remaining bias to report:* for 2021–2024 about 20% of games are unlabeled, and they are
probably disproportionately small releases, so the labeled sample is slightly skewed toward
games that got noticed.

## 4. Remove apps that are not standalone games

- **Playtests and demos** are listed as separate apps. They are free test builds, not products.
- **Software** (Utilities, Design & Illustration, Video Production, ...) is sold on Steam too.
  An app is dropped when it has a software genre and none of the core game genres.

In [7]:
name_l = df["name"].fillna("").str.lower()
is_test_build = name_l.str.contains(r"\bplaytest\b|\bdemo\b")
print(f"Playtest/demo apps: {is_test_build.sum():,}")

SOFTWARE_GENRES = {"Utilities", "Design & Illustration", "Animation & Modeling", "Video Production",
                   "Game Development", "Photo Editing", "Audio Production", "Software Training",
                   "Web Publishing", "Accounting"}
GAME_GENRES = {"Action", "Adventure", "Casual", "RPG", "Simulation", "Strategy", "Sports", "Racing",
               "Massively Multiplayer"}
is_software = df["genres"].apply(lambda L: bool(SOFTWARE_GENRES & set(L)) and not (GAME_GENRES & set(L)))
print(f"Software (non-game) apps: {is_software.sum():,}")

before = len(df)
games = df[
    ~is_test_build
    & ~is_software
    & (df["estimated_owners"] != "0 - 0")
    & df["release_dt"].notna()
    & (df["release_dt"] <= RELEASE_CUTOFF)
].copy()
print(f"All apps: {before:,}  ->  labeled games released <= {RELEASE_CUTOFF.date()}: {len(games):,}")
print(f"Release years: {games['release_dt'].dt.year.min()}-{games['release_dt'].dt.year.max()}")

Playtest/demo apps: 8,212
Software (non-game) apps: 1,449


All apps: 141,900  ->  labeled games released <= 2024-06-30: 76,727
Release years: 1997-2024


## 5. Leakage audit — which columns exist before launch?

Same rule as `old_analysis/week3/04`: a feature is allowed only if it would exist on the store
page on release day.

In [8]:
audit = pd.DataFrame([
    ("estimated_owners", "TARGET", "what we predict"),
    ("price (list price from package text)", "pre-release", "set by developer; *current* price is not used, it reflects later discounts"),
    ("price_current, discount", "post-release", "current price after later price cuts / sales"),
    ("release_date", "pre-release", "announced"),
    ("required_age", "pre-release", "set by developer"),
    ("supported / full-audio languages", "pre-release", "store page (can be extended later — mild risk)"),
    ("windows / mac / linux", "pre-release", "store page"),
    ("genres", "pre-release", "chosen by developer"),
    ("categories (single/multi-player, co-op, controller...)", "pre-release", "chosen by developer, EXCEPT the two below"),
    ("category: Steam Trading Cards", "post-release", "Valve only grants cards after a game passes a sales/confidence bar"),
    ("category: Family Sharing", "post-release", "Steam-wide feature added in 2024, applied retroactively"),
    ("category: Remote Play (Together / on TV / Phone / Tablet)", "uncertain", "Remote Play Together launched 2019 and was applied to existing games"),
    ("category: Steam Workshop", "uncertain", "mod support is usually added after a game has an audience"),
    ("screenshots (count)", "pre-release", "store page (can be updated later — mild risk)"),
    ("short / detailed description", "pre-release", "store page text"),
    ("website, support email", "pre-release", "store page"),
    ("developers / publishers", "pre-release", "known at announcement; history computed on an expanding window"),
    ("achievements (count)", "uncertain", "usually set at launch but can be added in updates"),
    ("tags (+ vote counts)", "post-release", "user-voted; number of tags and votes grow with player count"),
    ("dlc_count", "post-release", "DLC is released after the base game"),
    ("positive / negative / user_score / recommendations", "post-release", "reviews — same leak as User_Count in Phase 1"),
    ("metacritic_score", "launch-window", "like Critic_Score in old Phase 2 Tier 2; test separately"),
    ("peak_ccu, playtime", "post-release", "measured on players"),
    ("movies (trailers)", "unavailable", "empty for every app in this version"),
], columns=["column", "status", "reason"])
audit

,column,status,reason
0,estimated_owners,TARGET,what we predict
1,price (list price from package text),pre-release,"set by developer; *current* price is not used,..."
2,"price_current, discount",post-release,current price after later price cuts / sales
3,release_date,pre-release,announced
4,required_age,pre-release,set by developer
5,supported / full-audio languages,pre-release,store page (can be extended later — mild risk)
6,windows / mac / linux,pre-release,store page
7,genres,pre-release,chosen by developer
8,"categories (single/multi-player, co-op, contro...",pre-release,"chosen by developer, EXCEPT the two below"
9,category: Steam Trading Cards,post-release,Valve only grants cards after a game passes a ...


## 6. Build pre-release features

In [9]:
def list_price(row):
    # Package text looks like "Game - $14.99 $5.24" (list, then discounted) or "Game - $4.99".
    # The first dollar amount is the list price. Fall back to the current price.
    txt = row["package_text"]
    if isinstance(txt, str):
        m = re.search(r"\$(\d+(?:\.\d+)?)", txt)
        if m:
            return float(m.group(1))
    return float(row["price_current"] or 0.0)


g = games
g["price"] = g.apply(list_price, axis=1)
g["is_free"] = (g["price"] == 0).astype(int)
g["log_price"] = np.log1p(g["price"])
print(f"List price differs from current price for {(g['price'] != g['price_current']).mean():.1%} of games")

g["year"] = g["release_dt"].dt.year
g["month"] = g["release_dt"].dt.month
g["n_languages"] = g["supported_languages"].str.len()
g["n_audio_languages"] = g["full_audio_languages"].str.len()
g["has_english"] = g["supported_languages"].apply(lambda L: int(any("English" in x for x in L)))
g["n_platforms"] = g[["windows", "mac", "linux"]].astype(int).sum(axis=1)
for c in ["mac", "linux"]:
    g[c] = g[c].astype(int)
g["has_website"] = (g["website"].str.len() > 0).astype(int)
g["has_support_email"] = (g["support_email"].str.len() > 0).astype(int)
g["short_desc_len"] = g["short_description"].str.len()
g["log_detailed_desc_len"] = np.log1p(g["detailed_description_len"])
g["n_developers"] = g["developers"].str.len()
g["self_published"] = g.apply(
    lambda r: int(bool(r["developers"]) and set(r["developers"]) == set(r["publishers"])), axis=1)

POST_RELEASE_CATEGORIES = {"Steam Trading Cards", "Family Sharing"}
UNCERTAIN_CATEGORIES = {"Steam Workshop", "Remote Play Together", "Remote Play on TV",
                        "Remote Play on Phone", "Remote Play on Tablet"}
g["categories_pre"] = g["categories"].apply(
    lambda L: [c for c in L if c not in POST_RELEASE_CATEGORIES | UNCERTAIN_CATEGORIES])
g["cat_Steam_Workshop"] = g["categories"].apply(lambda L: int("Steam Workshop" in L))
g["cat_Remote_Play_any"] = g["categories"].apply(lambda L: int(any(c.startswith("Remote Play") for c in L)))
g["n_categories"] = g["categories_pre"].str.len()

List price differs from current price for 51.4% of games


Multi-hot encode genres and the most common (pre-release) categories.

In [10]:
def multi_hot(series, prefix, top_n):
    counts = series.explode().value_counts()
    keep = counts.head(top_n).index
    out = pd.DataFrame(index=series.index)
    for k in keep:
        col = prefix + re.sub(r"[^0-9a-zA-Z]+", "_", k).strip("_")
        out[col] = series.apply(lambda L, k=k: int(k in L))
    return out, counts


genre_df, genre_counts = multi_hot(g["genres"], "genre_", 20)
cat_df, cat_counts = multi_hot(g["categories_pre"], "cat_", 20)
g = pd.concat([g, genre_df, cat_df], axis=1)
print("Genres kept:", list(genre_df.columns))
print("Categories kept:", list(cat_df.columns))

Genres kept: ['genre_Indie', 'genre_Casual', 'genre_Action', 'genre_Adventure', 'genre_Simulation', 'genre_Strategy', 'genre_RPG', 'genre_Early_Access', 'genre_Free_To_Play', 'genre_Sports', 'genre_Racing', 'genre_Massively_Multiplayer', 'genre_Violent', 'genre_Gore', 'genre_Education', 'genre_Nudity', 'genre_Sexual_Content', 'genre_Utilities', 'genre_Design_Illustration', 'genre_Game_Development']
Categories kept: ['cat_Single_player', 'cat_Steam_Achievements', 'cat_Steam_Cloud', 'cat_Full_controller_support', 'cat_Multi_player', 'cat_Partial_Controller_Support', 'cat_PvP', 'cat_Co_op', 'cat_Steam_Leaderboards', 'cat_Online_PvP', 'cat_Shared_Split_Screen', 'cat_Tracked_Controller_Support', 'cat_VR_Only', 'cat_Online_Co_op', 'cat_Shared_Split_Screen_PvP', 'cat_Stats', 'cat_Shared_Split_Screen_Co_op', 'cat_Cross_Platform_Multiplayer', 'cat_Includes_level_editor', 'cat_In_App_Purchases']


### Developer / publisher track record (expanding window)

For a game released on date *D*, count only the same developer's (first-listed) games released
**strictly before D**, exactly as in `old_analysis/week3/04`:

- `dev_prior_games` — how many games the developer shipped before. **Fully leakage-free.**
- `dev_prior_hit_rate`, `dev_prior_max_log_owners` — how well those earlier games did.

*Caveat for the write-up:* the earlier games' owner counts are measured at the dataset snapshot
(2025), not as of date *D*, so they include sales that happened after *D*. The modeling
notebook should report results with and without these two outcome-based history features.

In [11]:
g["owners_mid"] = (g["owners_low"] + g["owners_high"]) / 2
g["log_owners_mid"] = np.log10(g["owners_mid"])
g["is_hit"] = (g["owners_low"] >= HIT_OWNERS).astype(int)
g["developer"] = g["developers"].apply(lambda L: L[0] if L else None)
g["publisher"] = g["publishers"].apply(lambda L: L[0] if L else None)


def prior_history(frame, key, prefix):
    # Aggregate per (key, date), take cumulative sums, then subtract the current date's own
    # contribution so same-day releases never see each other.
    sub = frame[frame[key].notna()]
    per_day = (
        sub.groupby([key, "release_dt"])
        .agg(n=("appid", "size"), hits=("is_hit", "sum"), mx=("log_owners_mid", "max"))
        .reset_index()
        .sort_values([key, "release_dt"])
    )
    grp = per_day.groupby(key)
    per_day[f"{prefix}_prior_games"] = grp["n"].cumsum() - per_day["n"]
    per_day["_prior_hits"] = grp["hits"].cumsum() - per_day["hits"]
    per_day[f"{prefix}_prior_max_log_owners"] = grp["mx"].transform(lambda s: s.cummax().shift(1))
    per_day[f"{prefix}_prior_hit_rate"] = per_day["_prior_hits"] / per_day[f"{prefix}_prior_games"].replace(0, np.nan)
    cols = [key, "release_dt", f"{prefix}_prior_games", f"{prefix}_prior_hit_rate", f"{prefix}_prior_max_log_owners"]
    out = frame.merge(per_day[cols], on=[key, "release_dt"], how="left")
    out[f"{prefix}_prior_games"] = out[f"{prefix}_prior_games"].fillna(0)
    return out


g = prior_history(g, "developer", "dev")
g = prior_history(g, "publisher", "pub")
print(f"Games whose developer shipped something earlier: {(g['dev_prior_games'] > 0).mean():.1%}")

Games whose developer shipped something earlier: 39.6%


**Leakage spot-check:** trace one prolific developer chronologically; the first game must show
no history and each count must only reflect earlier releases.

In [12]:
top_dev = g["developer"].value_counts().index[0]
g[g["developer"] == top_dev].sort_values("release_dt")[
    ["name", "release_dt", "estimated_owners", "dev_prior_games", "dev_prior_hit_rate", "dev_prior_max_log_owners"]
].head(8)

,name,release_dt,estimated_owners,dev_prior_games,dev_prior_hit_rate,dev_prior_max_log_owners
55175,Heroes Rise: The Prodigy,2014-06-17,20000 - 50000,0.0,NaN,NaN
45911,Heroes Rise: The Hero Project,2014-06-17,0 - 20000,0.0,NaN,NaN
52927,Heroes Rise: HeroFall,2014-07-11,0 - 20000,2.0,0.0,4.544068
39336,Choice of the Deathless,2014-08-29,0 - 20000,3.0,0.0,4.544068
45531,Thieves' Gambit: The Curse of the Black Cat,2014-10-17,0 - 20000,4.0,0.0,4.544068
58802,Mecha Ace,2014-12-05,0 - 20000,5.0,0.0,4.544068
22289,Choice of Robots,2014-12-19,50000 - 100000,6.0,0.0,4.544068
25151,Psy High,2014-12-22,0 - 20000,7.0,0.0,4.875061


### Derived indicators

Three more pre-release indicators computed from the same download, no extra data needed.

**(a) Store-page lead time, estimated from AppID.** Steam hands out AppIDs in increasing order
when a developer creates a store page, so the AppID dates the page. For each AppID, the page
creation date is estimated as a low (2nd) percentile of the release dates of the next 1,000 apps
created after it, made monotone. `lead_days` = release date − estimated creation date, a proxy for
how long the game was marketed before launch. Computed over **all** 141,900 apps (labeled or
not), since every app's release date helps date the AppID sequence. Leads below −30 days are
re-releases that kept an old release date and are set to missing; small negatives are clipped
to 0.

**(b) Launch competition.** Number of non-test apps released in the same week, and in the same
first-listed genre in the same month. Release schedules are public before launch.

**(c) Title features.** Title length, a sequel marker (trailing number or Roman numeral), and a
subtitle (`:` or ` - `).

In [13]:
# (a) AppID -> estimated store-page creation date
seq = df[["appid", "release_dt"]].dropna().sort_values("appid").reset_index(drop=True)
days = (seq["release_dt"] - pd.Timestamp("1970-01-01")).dt.days.astype(float)
fwd_low = days[::-1].rolling(1000, min_periods=50).quantile(0.02)[::-1]
created = fwd_low[::-1].cummin()[::-1]
seq["page_created_est"] = pd.Timestamp("1970-01-01") + pd.to_timedelta(created, unit="D")
g = g.merge(seq[["appid", "page_created_est"]], on="appid", how="left")
lead = (g["release_dt"] - g["page_created_est"]).dt.days
g["lead_days"] = lead.where(lead >= -30).clip(lower=0)
g["log_lead_days"] = np.log1p(g["lead_days"])
print("Estimated store-page lead time (days):")
print(g["lead_days"].describe(percentiles=[.25, .5, .75, .95]).round(0).to_string())

# (b) launch competition, counted over all non-test apps (labeled or not)
pool = df.loc[~is_test_build & ~is_software & df["release_dt"].notna(), ["release_dt", "genres"]].copy()
pool["week"] = pool["release_dt"].dt.to_period("W")
pool["ym"] = pool["release_dt"].dt.to_period("M")
pool["genre1"] = pool["genres"].apply(lambda L: L[0] if L else "None")
week_n = pool.groupby("week").size().rename("same_week_releases")
genre_month_n = pool.groupby(["genre1", "ym"]).size().rename("same_genre_month_releases")
g["week"] = g["release_dt"].dt.to_period("W")
g["ym"] = g["release_dt"].dt.to_period("M")
g["genre1"] = g["genres"].apply(lambda L: L[0] if L else "None")
g = g.merge(week_n, left_on="week", right_index=True, how="left")
g = g.merge(genre_month_n, left_on=["genre1", "ym"], right_index=True, how="left")
g = g.drop(columns=["week", "ym"])

# (c) title features
title = g["name"].fillna("")
g["title_words"] = title.str.split().str.len()
g["has_sequel_marker"] = title.str.contains(r"(?:\s|^)(?:[2-9]|1[0-9]|II|III|IV|V|VI|VII|VIII|IX|X)(?:\s*$|\s*:)").astype(int)
g["has_subtitle"] = title.str.contains(r":|\s-\s").astype(int)
print(f"\nSequel marker: {g['has_sequel_marker'].mean():.1%} | subtitle: {g['has_subtitle'].mean():.1%}")

Estimated store-page lead time (days):
count    76259.0
mean       198.0
std        343.0
min          0.0
25%         10.0
50%         56.0
75%        227.0
95%        895.0
max       5739.0



Sequel marker: 4.6% | subtitle: 18.4%


## 7. Target distribution

In [14]:
OWNER_CLASSES = [
    ("<20k", 0, 20_000), ("20k-100k", 20_000, 100_000),
    ("100k-1M", 100_000, 1_000_000), ("1M+", 1_000_000, np.inf),
]
def owner_class(low):
    for i, (_, lo, hi) in enumerate(OWNER_CLASSES):
        if lo <= low < hi:
            return i
g["owner_class"] = g["owners_low"].apply(owner_class)

dist = g["owner_class"].value_counts().sort_index()
dist.index = [c[0] for c in OWNER_CLASSES]
print("4-class ordinal target (share of games):")
print((dist / len(g)).round(3).to_string())
print(f"\nBinary target is_hit (>= {HIT_OWNERS:,} owners): {g['is_hit'].mean():.1%} positive "
      f"({g['is_hit'].sum():,} games)")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(dist.index, dist.values, color="#4C72B0")
ax.set_yscale("log")
ax.set_ylabel("games (log scale)")
ax.set_title("Owner class distribution (labeled games, released <= 2024-06)")
for i, v in enumerate(dist.values):
    ax.text(i, v * 1.1, f"{v:,}", ha="center")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "01_owner_class_distribution.png"), dpi=150)
print("Saved outputs/01_owner_class_distribution.png")

4-class ordinal target (share of games):
<20k        0.697
20k-100k    0.197
100k-1M     0.090
1M+         0.017

Binary target is_hit (>= 100,000 owners): 10.7% positive (8,194 games)


Saved outputs/01_owner_class_distribution.png


The target is extremely skewed: most Steam games sell fewer than 20k copies. As in the old
Phase 2, the primary target is a **binary hit** (≥100k owners), evaluated with ROC-AUC and
average precision against the base rate. The 4-class version is kept for an ordinal model later.

## 8. Do the pre-release features carry signal?

Hit rate across each candidate feature, versus the overall base rate. If hit rate doesn't move,
there is nothing for a model to learn.

In [15]:
base = g["is_hit"].mean()
print(f"Base hit rate: {base:.2%}\n")


def hit_rate_by(col, bins=None, labels=None):
    key = pd.cut(g[col], bins=bins, labels=labels, include_lowest=True) if bins is not None else g[col]
    return g.groupby(key, observed=True)["is_hit"].agg(hit_rate="mean", games="size").round(3)


print("By list price:");      print(hit_rate_by("price", [-0.01, 0, 5, 10, 20, 40, 1000], ["free", "<=5", "5-10", "10-20", "20-40", ">40"]), "\n")
print("By # languages:");     print(hit_rate_by("n_languages", [0, 1, 3, 8, 15, 200], ["1", "2-3", "4-8", "9-15", "16+"]), "\n")
print("By # screenshots:");   print(hit_rate_by("n_screenshots", [-1, 5, 10, 20, 500], ["<=5", "6-10", "11-20", "21+"]), "\n")
print("By developer history:"); print(hit_rate_by("dev_prior_games", [-1, 0, 2, 10, 10_000], ["first game", "1-2", "3-10", "11+"]), "\n")
print("By self-published:");  print(hit_rate_by("self_published"), "\n")
print("By store-page lead time (days):"); print(hit_rate_by("lead_days", [-1, 14, 60, 180, 365, 10_000], ["<=2wk", "2wk-2mo", "2-6mo", "6-12mo", ">1yr"]), "\n")
print("By sequel marker:");   print(hit_rate_by("has_sequel_marker"))

Base hit rate: 10.68%

By list price:
       hit_rate  games
price                 
free      0.166   8347
<=5       0.048  33928
5-10      0.083  17808
10-20     0.182  12476
20-40     0.349   3359
>40       0.331    809 

By # languages:
             hit_rate  games
n_languages                 
1               0.058  41532
2-3             0.066  13876
4-8             0.186  10780
9-15            0.311   7363
16+             0.179   3176 

By # screenshots:
               hit_rate  games
n_screenshots                 
<=5               0.057  19745
6-10              0.097  38215
11-20             0.175  16059
21+               0.198   2708 

By developer history:
                 hit_rate  games
dev_prior_games                 
first game          0.097  46310
1-2                 0.126  15510
3-10                0.145   9385
11+                 0.075   5522 

By self-published:
                hit_rate  games
self_published                 
0                  0.172  24134
1           

In [16]:
flag_cols = list(genre_df.columns) + list(cat_df.columns) + ["mac", "linux", "has_website", "is_free"]
lift = pd.DataFrame({
    "hit_rate_if_1": [g.loc[g[c] == 1, "is_hit"].mean() for c in flag_cols],
    "games_with_flag": [int(g[c].sum()) for c in flag_cols],
}, index=flag_cols)
lift["lift_vs_base"] = lift["hit_rate_if_1"] / base
lift = lift.sort_values("lift_vs_base")

fig, ax = plt.subplots(figsize=(8, 10))
colors = ["#C44E52" if v < 1 else "#55A868" for v in lift["lift_vs_base"]]
ax.barh(lift.index, lift["lift_vs_base"], color=colors)
ax.axvline(1, color="black", lw=0.8)
ax.set_xlabel("hit rate when flag = 1, relative to base rate")
ax.set_title("Pre-release flags: lift in hit rate")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "01_feature_lift.png"), dpi=150)
print("Saved outputs/01_feature_lift.png")
lift.round(3).tail(10)

Saved outputs/01_feature_lift.png


,hit_rate_if_1,games_with_flag,lift_vs_base
cat_Steam_Cloud,0.202,20129,1.894
cat_Multi_player,0.211,14127,1.974
genre_Free_To_Play,0.228,4783,2.134
cat_Co_op,0.230,7349,2.151
cat_Online_PvP,0.230,6018,2.152
cat_Includes_level_editor,0.257,1964,2.408
genre_Massively_Multiplayer,0.268,1591,2.513
cat_Online_Co_op,0.282,4063,2.637
cat_Cross_Platform_Multiplayer,0.298,2091,2.790
cat_In_App_Purchases,0.418,1525,3.911


## 9. Save the modeling table

Only pre-release features plus targets and keys. Post-release columns are dropped here so they
cannot slip into a model by accident. `metacritic_score` is kept separately as the launch-window
"Tier 2" feature, like `Critic_Score` in the old analysis.

In [17]:
KEYS = ["appid", "name", "developer", "publisher", "release_dt", "year", "month"]
TARGETS = ["estimated_owners", "owners_low", "log_owners_mid", "owner_class", "is_hit"]
PRE_RELEASE = [
    "price", "log_price", "is_free", "required_age",
    "n_languages", "n_audio_languages", "has_english", "n_platforms", "mac", "linux",
    "n_screenshots", "has_website", "has_support_email", "short_desc_len", "log_detailed_desc_len",
    "n_developers", "self_published", "n_categories",
    "dev_prior_games", "pub_prior_games",
    "lead_days", "log_lead_days", "same_week_releases", "same_genre_month_releases",
    "title_words", "has_sequel_marker", "has_subtitle",
] + list(genre_df.columns) + list(cat_df.columns)
OUTCOME_HISTORY = ["dev_prior_hit_rate", "dev_prior_max_log_owners", "pub_prior_hit_rate", "pub_prior_max_log_owners"]
UNCERTAIN = ["achievements", "cat_Steam_Workshop", "cat_Remote_Play_any"]
LAUNCH_WINDOW = ["metacritic_score"]
TEXT = ["short_description"]

table = g[KEYS + TARGETS + PRE_RELEASE + OUTCOME_HISTORY + UNCERTAIN + LAUNCH_WINDOW + TEXT]
table.to_csv(os.path.join(DATA_DIR, "steam_prerelease_features.csv.gz"), index=False)

pd.Series({
    "PRE_RELEASE": PRE_RELEASE, "OUTCOME_HISTORY": OUTCOME_HISTORY,
    "UNCERTAIN": UNCERTAIN, "LAUNCH_WINDOW": LAUNCH_WINDOW, "TEXT": TEXT,
}).to_json(os.path.join(DATA_DIR, "feature_groups.json"))

print(f"Saved data/processed/steam_prerelease_features.csv.gz  ({table.shape[0]:,} rows x {table.shape[1]} cols)")
print(f"  pre-release features: {len(PRE_RELEASE)} | outcome-history: {len(OUTCOME_HISTORY)} | "
      f"metacritic present: {(g['metacritic_score'] > 0).mean():.1%} of games")

Saved data/processed/steam_prerelease_features.csv.gz  (76,727 rows x 88 cols)
  pre-release features: 67 | outcome-history: 4 | metacritic present: 5.2% of games


## Summary and next step

- **Data:** Steam JSON (the CSV header is broken). Trailer data is empty in this version.
- **Label:** SteamSpy owner ranges. `"0 - 0"` is a missing label, not zero sales, and labels
  stop updating around mid-2025, so the modeling set is labeled games released up to 2024-06-30.
- **Scope:** playtests, demos and non-game software removed.
- **Leakage audit:** reviews, playtime, CCU, tags, DLC count, current price, Trading Cards and
  Family Sharing are excluded as post-release; achievements, Workshop and Remote Play are held
  out as "uncertain" for a separate ablation.
- **Derived indicators:** store-page lead time (from AppID order), launch-week and genre-month
  competition, title features.
- **Target:** binary hit (≥100k owners) as primary; 4-class ordinal kept for later.

**Next (`02_baseline_models.ipynb`):** Logistic Regression and Random Forest on the pre-release
features, with a chronological split (train ≤2021, test 2022–mid-2024) as the headline, and
ablations: (a) without outcome-based developer history, (b) plus the "uncertain" features,
(c) plus `metacritic_score`, (d) plus description text (TF-IDF).